# Validation summary: peak loads of the mode I DCB references

Collects the peak loads of the validation notebooks into one table, the table of peak loads of Section *Validation against the literature* of
`doc/source/cohesive_zone.rst`:

- the `panels` curves are read from `results/*.npz`, produced by the notebooks
  `dcb_alfano2001.ipynb`, `dcb_camanho2003.ipynb`, `dcb_turon2007.ipynb`, `dcb_krueger2008.ipynb`, `dcb_tijs2022.ipynb` and `dcb_tijs2023_phd.ipynb`;
- the FE curves and the experimental peaks are copied from the dictionaries of those notebooks, where their sources and
  the digitisation uncertainty are given;
- the benchmark of Krueger (2008) is not an FE curve but the delamination onset criterion of his Eq. (5), evaluated from the
  loads of his Figs. 13 and 36 and the energy release rates of his Figs. 11 and 34; it is entered in the FE column;
- the reference of linear elastic fracture mechanics is the corrected beam theory (CBT), `dcb_utils.lefm_curve(..., corrected=True)`.

The initial compliance of the table of initial compliances of the same section is computed in `dcb_lecinana2023.ipynb` and `dcb_tijs2023_phd.ipynb`.

Run this notebook from the `notebooks` folder.

In [1]:
import numpy as np
from IPython.display import Markdown, display

import dcb_utils as du

R = 'results'
# Alfano and Crisfield plot the displacement of one arm, u, and delta = 2u
u2d = lambda u: 2*np.asarray(u, dtype=float)
# AS4D/PEKK-FC, average of the tension and compression moduli (Tijs et al. 2022)
pekk = dict(E1=(138300. + 128000.)/2, E2=(10400. + 11500.)/2, nu12=0.316, G12=5190.)
# Krueger (2008) benchmark, Eq. (5): P1 (N) and GI (J/m2) at delta/2 = 1 mm, see
# dcb_krueger2008.ipynb
def kbench(P1, GI, G1c):
    s = np.sqrt(G1c*1000./np.asarray(GI, dtype=float))
    return np.r_[0., 2*s], np.r_[0., np.asarray(P1, dtype=float)*s]


k_ud = dict(E1=139400., E2=10160., nu12=0.30, G12=4600., h=1.5, b=25., L=150.,
            a0=30., G1c=0.1703)
k_md_arm = [30, -30, 0, -30, 0, 30, 0, 0, 0, 0, 30, 0, -30, 0, -30, 30]
k_md = dict(E1=146900., E2=10600., nu12=0.33, G12=5450., stack=k_md_arm, plyt=0.125,
            b=25.4, L=177.8, a0=31., G1c=0.3405)
# quasi-isotropic laminate of the welded specimens (Tijs 2023)
qi = [0, 45, 90, -45, 0, -45, 90, 45]
qi = qi + qi[::-1]

Cases: label, specimen for the LEFM curve, results file, FE curve `(delta, P)` or `None`, experimental peak or `None`.
Superscripts of the table: *d* digitised from a figure, *t* from a table of the reference, *m* mean of the digitised peaks of several tests; *n* not used to validate the model, see the note in `dcb_camanho2003.ipynb`.

In [2]:
cases = [
    ('Alfano 2001, Table I, $t_o$ = 57 MPa',
     dict(E1=135300., E2=9000., nu12=0.24, G12=5200., h=1.5, b=20., L=100., a0=30., G1c=0.28),
     'alfano2001_TableI_to57', (u2d([0, 1.0, 1.5, 2.0, 3.0, 4.0, 4.8]), np.array([0, 61.5, 52., 44., 35., 30.5, 28.])), 'd', None, None),
    ('Alfano 2001, Table I, $t_o$ = 1.7 MPa',
     dict(E1=135300., E2=9000., nu12=0.24, G12=5200., h=1.5, b=20., L=100., a0=30., G1c=0.28),
     'alfano2001_TableI_to1.7', (u2d([0, 0.5, 0.9, 1.2, 1.45, 2.0]), np.array([0, 24., 37., 45., 49.5, 46.])), 'd', None, None),
    ('Alfano 2001, XAS-913C',
     dict(E1=126000., E2=7500., nu12=0.263, G12=4981., h=1.5, b=30., L=100., a0=30., G1c=0.281),
     'alfano2001_TableII_XAS913C', (u2d([0, 1.1, 1.5, 2.0]), np.array([0, 90., 80., 70.])), 'd', None, None),
    ('Camanho 2003, AS4/PEEK',
     dict(E1=122700., E2=10100., nu12=0.25, G12=5500., h=1.56, b=25.4, L=102., a0=32.9, G1c=0.969),
     'camanho2003_DCB_AS4PEEK', (np.array([0, 4.0, 5.0, 5.3, 5.8]), np.array([0, 116., 150., 155.3, 143.])), 't', 147.1, 't'),
    ('Turon 2007, T300/977-2',
     dict(E1=150000., E2=11000., nu12=0.25, G12=6000., h=1.98, b=20., L=150., a0=55., G1c=0.352),
     'turon2007_le0.125_tau60', (np.array([0, 4.0, 5.0, 5.3, 6.0]), np.array([0, 44., 55., 57., 53.])), 'd', 62., 'd'),
    ('Krueger 2008, UD $[0]_{24}$',
     k_ud, 'krueger2008_DCB_UD_tau40',
     kbench([81.0, 74.0, 68.0, 62.5, 57.5, 52.5, 36.7],
            [303., 270., 240., 215., 192., 170., 105.], 0.1703), 'd', None, None),
    ('Krueger 2008, D$\\pm$30',
     k_md, 'krueger2008_DCB_MD_tau40',
     kbench([128.0, 117.5, 108.0, 99.5, 91.5, 63.0],
            [532., 478., 425., 378., 342., 212.], 0.3405), 'd', None, None),
    ('Tijs 2022, $a_0$ = 48 mm',
     dict(pekk, h=2.1, b=25., L=100., a0=48., G1c=1.12),
     'tijs2022_a48', (np.array([0, 5.7, 6.5, 7.5]), np.array([0, 167., 152., 138.])), 'd', 142., 'd'),
    ('Tijs 2022, $a_0$ = 40 mm, 0.95 N/mm',
     dict(pekk, h=2.1, b=25., L=100., a0=40., G1c=0.95),
     'tijs2022_a40_G0.95', None, None, 161., 'd'),
    ('Tijs 2023, full weld, 2.1 N/mm',
     dict(pekk, stack=qi, plyt=0.14, b=25., bond_width=15., L=144.3, a0=44.3, G1c=2.1),
     'tijs2023_full_weld', (np.array([0, 6.2, 7., 8.]), np.array([0, 155., 149., 141.])), 'd', 160., 'm'),
    ('Tijs 2023, weld center, 2.1 N/mm',
     dict(pekk, stack=qi, plyt=0.14, b=12.7, L=136.2, a0=36.2, G1c=2.1),
     'tijs2023_weld_center', (np.array([0, 5.0, 6., 7.]), np.array([0, 128., 121., 113.])), 'd', 129., 'm'),
    ('Tijs 2023, weld center, 1.95 N/mm',
     dict(pekk, stack=qi, plyt=0.14, b=12.7, L=136.2, a0=36.2, G1c=1.95),
     'tijs2023_weld_center_standard', (np.array([0, 5.3, 6., 7.]), np.array([0, 122., 118., 111.])), 'd', 123., 'd'),
    ('Tijs 2023, weld center, 1.12 N/mm',
     dict(pekk, stack=qi, plyt=0.14, b=12.7, L=136.2, a0=36.2, G1c=1.12),
     'tijs2023_weld_center_autoclave', (np.array([0, 4.5, 5., 6.]), np.array([0, 96., 94., 86.])), 'd', None, None),
]

In [3]:
rows = []
for label, case, fname, fe, fe_src, exp_peak, exp_src in cases:
    r = du.load(f'{R}/{fname}.npz')
    pan = du.summary(np.r_[0, r['delta']], np.r_[0, r['P']])
    # the LEFM curve covers the crack growth of every run
    cbt = du.summary(*du.lefm_curve(case, case['a0'] + 60, corrected=True))
    row = dict(label=label, Pmax=pan['Pmax'], dPmax=pan['delta_Pmax'],
               cbt=cbt['Pmax'], dK0=100*(pan['K0']/cbt['K0'] - 1),
               growth=float(r['a'][-1] - case['a0']), bisections=int(r['failed']))
    if fe is not None:
        row['fe'], row['fe_src'] = du.summary(*fe)['Pmax'], fe_src
    if exp_peak is not None:
        row['exp'], row['exp_src'] = exp_peak, exp_src
    rows.append(row)

# cases kept for reference only, see the note of each notebook
NOTES = {'Camanho 2003, AS4/PEEK': 'n'}
diff = lambda a, b: 100*(a/b - 1)
lines = ['| case | panels $P_{max}$ | $\\delta$ | FE $P_{max}$ | diff. (%) | test $P_{max}$ | diff. (%) '
         '| CBT $P_{max}$ | diff. (%) | $\\Delta K_0$ (%) | $\\Delta a$ (mm) | bisections |',
         '|---|---|---|---|---|---|---|---|---|---|---|---|']
for r in rows:
    fe = (f"{r['fe']:.1f}<sup>{r['fe_src']}</sup> | {diff(r['Pmax'], r['fe']):+.1f}" if 'fe' in r else '- | -')
    ex = (f"{r['exp']:.1f}<sup>{r['exp_src']}</sup> | {diff(r['Pmax'], r['exp']):+.1f}" if 'exp' in r else '- | -')
    label = r['label'] + (f"<sup>{NOTES[r['label']]}</sup>" if r['label'] in NOTES else '')
    lines.append(f"| {label} | {r['Pmax']:.1f} | {r['dPmax']:.2f} | {fe} | {ex} | {r['cbt']:.1f} "
                 f"| {diff(r['Pmax'], r['cbt']):+.1f} | {r['dK0']:+.0f} | {r['growth']:.1f} | {r['bisections']} |")
display(Markdown('\n'.join(lines)))

| case | panels $P_{max}$ | $\delta$ | FE $P_{max}$ | diff. (%) | test $P_{max}$ | diff. (%) | CBT $P_{max}$ | diff. (%) | $\Delta K_0$ (%) | $\Delta a$ (mm) | bisections |
|---|---|---|---|---|---|---|---|---|---|---|---|
| Alfano 2001, Table I, $t_o$ = 57 MPa | 64.5 | 1.85 | 61.5<sup>d</sup> | +4.8 | - | - | 63.2 | +2.1 | +17 | 9.0 | 0 |
| Alfano 2001, Table I, $t_o$ = 1.7 MPa | 50.8 | 2.90 | 49.5<sup>d</sup> | +2.6 | - | - | 63.2 | -19.6 | -25 | 6.0 | 0 |
| Alfano 2001, XAS-913C | 93.6 | 1.87 | 90.0<sup>d</sup> | +4.0 | - | - | 91.6 | +2.1 | +17 | 9.1 | 0 |
| Camanho 2003, AS4/PEEK<sup>n</sup> | 140.1 | 4.09 | 155.3<sup>t</sup> | -9.8 | 147.1<sup>t</sup> | -4.8 | 138.9 | +0.9 | +14 | 7.4 | 0 |
| Turon 2007, T300/977-2 | 64.0 | 4.16 | 57.0<sup>d</sup> | +12.3 | 62.0<sup>d</sup> | +3.3 | 63.3 | +1.2 | +12 | 11.1 | 0 |
| Krueger 2008, UD $[0]_{24}$ | 63.5 | 1.43 | 60.7<sup>d</sup> | +4.6 | - | - | 62.3 | +2.0 | +18 | 8.5 | 0 |
| Krueger 2008, D$\pm$30 | 108.7 | 1.70 | 102.4<sup>d</sup> | +6.1 | - | - | 114.7 | -5.2 | +10 | 7.1 | 0 |
| Tijs 2022, $a_0$ = 48 mm | 166.2 | 5.66 | 167.0<sup>d</sup> | -0.5 | 142.0<sup>d</sup> | +17.1 | 164.5 | +1.1 | +14 | 10.1 | 0 |
| Tijs 2022, $a_0$ = 40 mm, 0.95 N/mm | 182.1 | 3.71 | - | - | 161.0<sup>d</sup> | +13.1 | 179.1 | +1.6 | +17 | 10.6 | 0 |
| Tijs 2023, full weld, 2.1 N/mm | 148.0 | 6.41 | 155.0<sup>d</sup> | -4.5 | 160.0<sup>m</sup> | -7.5 | 155.5 | -4.8 | +4 | 4.9 | 0 |
| Tijs 2023, weld center, 2.1 N/mm | 119.2 | 5.82 | 128.0<sup>d</sup> | -6.9 | 129.0<sup>m</sup> | -7.6 | 123.0 | -3.1 | +8 | 5.5 | 0 |
| Tijs 2023, weld center, 1.95 N/mm | 115.0 | 5.58 | 122.0<sup>d</sup> | -5.7 | 123.0<sup>d</sup> | -6.5 | 118.5 | -3.0 | +8 | 5.8 | 0 |
| Tijs 2023, weld center, 1.12 N/mm | 87.9 | 4.13 | 96.0<sup>d</sup> | -8.5 | - | - | 89.8 | -2.2 | +9 | 8.2 | 0 |

Sensitivity to the penalty stiffness, Turon et al. (2007): $k_o = 10^6$ N/mm³, the value of the reference, against the $10^5$ N/mm³ used in all cases above. Alfano and Crisfield (2001), Table I, $t_o$ = 1.7 MPa: $k_o = 10^5$ N/mm³ against the $10^3$ N/mm³ of the run in the table above, both with 30 terms over the cohesive domain of 70 mm, see `dcb_alfano2001.ipynb`.

Sensitivity to the interfacial strength, Krueger (2008): the reference uses the virtual crack closure technique and gives no strength, so the two values $\tau_o = 40$ and 60 MPa are run; the 40 MPa runs are the ones in the table above.

In [4]:
for name in ('alfano2001_TableI_to1.7', 'alfano2001_TableI_to1.7_K1e5',
             'turon2007_le0.125_tau60', 'turon2007_le0.125_tau60_K1e6',
             'krueger2008_DCB_UD_tau40', 'krueger2008_DCB_UD_tau60',
             'krueger2008_DCB_MD_tau40', 'krueger2008_DCB_MD_tau60'):
    r = du.load(f'{R}/{name}.npz')
    s = du.summary(np.r_[0, r['delta']], np.r_[0, r['P']])
    print(f"{name}: Pmax = {s['Pmax']:.2f} N at delta = {s['delta_Pmax']:.2f} mm")

alfano2001_TableI_to1.7: Pmax = 50.78 N at delta = 2.90 mm
alfano2001_TableI_to1.7_K1e5: Pmax = 50.81 N at delta = 2.90 mm
turon2007_le0.125_tau60: Pmax = 64.02 N at delta = 4.16 mm
turon2007_le0.125_tau60_K1e6: Pmax = 64.13 N at delta = 4.19 mm
krueger2008_DCB_UD_tau40: Pmax = 63.52 N at delta = 1.43 mm
krueger2008_DCB_UD_tau60: Pmax = 64.24 N at delta = 1.39 mm
krueger2008_DCB_MD_tau40: Pmax = 108.69 N at delta = 1.70 mm
krueger2008_DCB_MD_tau60: Pmax = 109.97 N at delta = 1.66 mm


Rows of the table of peak loads of `doc/source/cohesive_zone.rst`, in the format of its `list-table`.

In [5]:
sup = lambda v, s: f"{v:.1f} `^{{{s}}}`"
for r in rows:
    label = r['label'].replace('$', '`') + (f" `^{{{NOTES[r['label']]}}}`" if r['label'] in NOTES else '')
    cells = [label, f"{r['Pmax']:.1f}", f"{r['dPmax']:.2f}"]
    cells += [sup(r['fe'], r['fe_src']), f"{diff(r['Pmax'], r['fe']):+.1f}"] if 'fe' in r else ['--', '--']
    cells += [sup(r['exp'], r['exp_src']), f"{diff(r['Pmax'], r['exp']):+.1f}"] if 'exp' in r else ['--', '--']
    cells += [f"{r['cbt']:.1f}", f"{diff(r['Pmax'], r['cbt']):+.1f}", f"{r['dK0']:+.0f}", f"{r['growth']:.1f}"]
    print('    * - ' + '\n      - '.join(cells))

    * - Alfano 2001, Table I, `t_o` = 57 MPa
      - 64.5
      - 1.85
      - 61.5 `^{d}`
      - +4.8
      - --
      - --
      - 63.2
      - +2.1
      - +17
      - 9.0
    * - Alfano 2001, Table I, `t_o` = 1.7 MPa
      - 50.8
      - 2.90
      - 49.5 `^{d}`
      - +2.6
      - --
      - --
      - 63.2
      - -19.6
      - -25
      - 6.0
    * - Alfano 2001, XAS-913C
      - 93.6
      - 1.87
      - 90.0 `^{d}`
      - +4.0
      - --
      - --
      - 91.6
      - +2.1
      - +17
      - 9.1
    * - Camanho 2003, AS4/PEEK `^{n}`
      - 140.1
      - 4.09
      - 155.3 `^{t}`
      - -9.8
      - 147.1 `^{t}`
      - -4.8
      - 138.9
      - +0.9
      - +14
      - 7.4
    * - Turon 2007, T300/977-2
      - 64.0
      - 4.16
      - 57.0 `^{d}`
      - +12.3
      - 62.0 `^{d}`
      - +3.3
      - 63.3
      - +1.2
      - +12
      - 11.1
    * - Krueger 2008, UD `[0]_{24}`
      - 63.5
      - 1.43
      - 60.7 `^{d}`
      - +4.6
      - --
      - --
      - 6